# Loading libary

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.metrics import balanced_accuracy_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier



In [5]:
import os
print(os.listdir('/kaggle/input'))

['datasets', 'competitions']


In [6]:
dataset_path = "/kaggle/input/competitions"
print(os.listdir(dataset_path))

['playground-series-s6e6']


In [7]:
import os
dataset_path = "/kaggle/input/datasets/fedesoriano"
print(os.listdir(dataset_path))

['stellar-classification-dataset-sdss17']


## 1. Load Data (Competition + Original SDSS17)

In [8]:
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e6/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/playground-series-s6e6/test.csv')
orig  = pd.read_csv('/kaggle/input/datasets/fedesoriano/stellar-classification-dataset-sdss17/star_classification.csv')
train = pd.concat([train, orig], ignore_index=True)

print('Train shape:', train.shape)
print('Test shape:', test.shape)
print('Orig shape:', orig.shape)
print('Train columns:', train.columns.tolist())

Train shape: (577347, 12)
Test shape: (247435, 11)
Orig shape: (100000, 18)
Train columns: ['id', 'alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'spectral_type', 'galaxy_population', 'class']


## 2. Merge Original SDSS17 Data

In [9]:
# Rename 'class' column in orig to match competition
orig = orig.rename(columns={'class': 'class'})

# Keep only columns that exist in train
train_feature_cols = [c for c in train.columns if c not in ['id']]
orig_cols_available = [c for c in train_feature_cols if c in orig.columns]
print('Matching columns:', orig_cols_available)

orig_subset = orig[orig_cols_available].copy()
orig_subset['id'] = -1  # placeholder

# Align columns
for col in train.columns:
    if col not in orig_subset.columns:
        orig_subset[col] = np.nan

orig_subset = orig_subset[train.columns]

# Combine
train_full = pd.concat([train, orig_subset], ignore_index=True)
print('Combined train shape:', train_full.shape)
print('Class distribution:')
print(train_full['class'].value_counts())

Matching columns: ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'class']
Combined train shape: (677347, 12)
Class distribution:
class
GALAXY    436925
QSO       136104
STAR      104318
Name: count, dtype: int64


## 3. Label Encoding

In [10]:
le = LabelEncoder()
y = le.fit_transform(train_full['class'])
print('Classes:', le.classes_)
print('Encoded y unique:', np.unique(y))

Classes: ['GALAXY' 'QSO' 'STAR']
Encoded y unique: [0 1 2]


## 4. Feature Engineering

In [11]:
def create_features(df):
    df = df.copy()
    mag_cols = ['u', 'g', 'r', 'i', 'z']

    # ── Basic color indices ──────────────────────────────────────────────
    df['u_g'] = df['u'] - df['g']
    df['g_r'] = df['g'] - df['r']
    df['r_i'] = df['r'] - df['i']
    df['i_z'] = df['i'] - df['z']
    df['u_r'] = df['u'] - df['r']
    df['g_i'] = df['g'] - df['i']
    df['g_z'] = df['g'] - df['z']
    df['u_z'] = df['u'] - df['z']
    df['u_i'] = df['u'] - df['i']
    df['r_z'] = df['r'] - df['z']

    # ── Statistical features ─────────────────────────────────────────────
    df['flux_mean']   = df[mag_cols].mean(axis=1)
    df['flux_std']    = df[mag_cols].std(axis=1)
    df['flux_min']    = df[mag_cols].min(axis=1)
    df['flux_max']    = df[mag_cols].max(axis=1)
    df['flux_range']  = df['flux_max'] - df['flux_min']
    df['flux_median'] = df[mag_cols].median(axis=1)
    df['flux_skew']   = df[mag_cols].skew(axis=1)

    # ── Non-linear interactions ───────────────────────────────────────────
    df['r_over_g']    = df['r'] / (df['g'] + 1e-6)
    df['i_over_z']    = df['i'] / (df['z'] + 1e-6)
    df['u_over_r']    = df['u'] / (df['r'] + 1e-6)
    df['color_index'] = (df['u'] - df['g']) - (df['r'] - df['i'])

    df['u_z'] = df['u'] - df['z']
    df['r_z'] = df['r'] - df['z']
    df['redshift_mag'] = df['redshift'] * df['flux_mean']
    df['fiber_ratio'] = df['fiber_ID'] / (df['plate'] + 1)
    df['mjd_plate'] = df['mjd'] * df['plate']

    # ── Spectral slope (linear fit across bands) ─────────────────────────
    # Approximate wavelengths: u=3543, g=4770, r=6231, i=7625, z=9134 Angstrom
    wavelengths = np.array([3543, 4770, 6231, 7625, 9134])
    mags = df[mag_cols].values
    w_mean = wavelengths.mean()
    m_mean = mags.mean(axis=1, keepdims=True)
    slope_num   = ((wavelengths - w_mean) * (mags - m_mean)).sum(axis=1)
    slope_denom = ((wavelengths - w_mean) ** 2).sum()
    df['spectral_slope'] = slope_num / slope_denom

    # ── Redshift features ────────────────────────────────────────────────
    if 'redshift' in df.columns:
        df['redshift_sq']   = df['redshift'] ** 2
        df['redshift_abs']  = np.abs(df['redshift'])
        df['redshift_log']  = np.log1p(np.abs(df['redshift']))
        df['redshift_sign'] = np.sign(df['redshift'])
        df['redshift_bin']  = pd.cut(df['redshift'], bins=20, labels=False)
        # Interaction: redshift × color
        df['z_times_gr']   = df['redshift'] * df['g_r']
        df['z_times_ui']   = df['redshift'] * df['u_i']

    # ── Positional features ──────────────────────────────────────────────
    if 'alpha' in df.columns and 'delta' in df.columns:
        df['alpha_delta']  = df['alpha'] * df['delta']
        df['alpha_sin']    = np.sin(np.radians(df['alpha']))
        df['alpha_cos']    = np.cos(np.radians(df['alpha']))
        df['delta_sin']    = np.sin(np.radians(df['delta']))

    return df


drop_cols = ['id', 'class']
X    = create_features(train_full.drop(columns=drop_cols, errors='ignore'))
test_feat = create_features(test.drop(columns=['id'], errors='ignore'))


# Fill NaNs — numeric columns only (redshift_bin is categorical, skip it)
num_cols = X.select_dtypes(include=[np.number]).columns
X[num_cols]         = X[num_cols].fillna(X[num_cols].median())
test_num_cols = test_feat.select_dtypes(include=[np.number]).columns
test_feat[test_num_cols] = test_feat[test_num_cols].fillna(test_feat[test_num_cols].median())

print('X shape:', X.shape)
print('test_feat shape:', test_feat.shape)

X shape: (677347, 43)
test_feat shape: (247435, 43)


## 5. Ordinal Encoding for Categorical Columns

In [12]:
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
print('Categorical columns:', cat_cols)

if cat_cols:
    encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X[cat_cols]         = encoder.fit_transform(X[cat_cols])
    test_feat[cat_cols] = encoder.transform(test_feat[cat_cols])

X         = X.astype(np.float32)
test_feat = test_feat.astype(np.float32)
print('Feature dtypes OK')

Categorical columns: ['spectral_type', 'galaxy_population']
Feature dtypes OK


## 6. Model Definitions (Tuned)

In [14]:
N_SPLITS   = 5
SEED       = 42
N_CLASSES  = len(le.classes_)

xgb = XGBClassifier(
    n_estimators=5000,
    learning_rate=0.005,
    max_depth=8,
    min_child_weight=2,
    subsample=0.9,
    colsample_bytree=0.9,
    gamma=0.1,
    reg_alpha=0.5,
    reg_lambda=2,
    tree_method='hist',
    eval_metric='mlogloss',
    random_state=42
)

lgb_params = dict(
    n_estimators        = 3000,
    learning_rate       = 0.02,
    num_leaves          = 127,
    max_depth           = -1,
    min_child_samples   = 20,
    subsample           = 0.85,
    subsample_freq      = 1,
    colsample_bytree    = 0.75,
    reg_alpha           = 0.1,
    reg_lambda          = 1.0,
    class_weight        = 'balanced',
    random_state        = SEED,
    n_jobs              = -1,
    verbose             = -1,
)

cat = CatBoostClassifier(
    iterations=5000,
    learning_rate=0.02,
    depth=8,
    loss_function='MultiClass',
    eval_metric='Accuracy',
    verbose=200
)
print('Models configured')

Models configured


## 7. OOF Training with Early Stopping

In [15]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_xgb  = np.zeros((len(X), N_CLASSES))
oof_lgb  = np.zeros((len(X), N_CLASSES))
oof_cat  = np.zeros((len(X), N_CLASSES))

test_xgb = np.zeros((len(test_feat), N_CLASSES))
test_lgb = np.zeros((len(test_feat), N_CLASSES))
test_cat = np.zeros((len(test_feat), N_CLASSES))

X_arr    = X.values
test_arr = test_feat.values

fold_scores_xgb = []
fold_scores_lgb = []
fold_scores_cat = []

for fold, (trn_idx, val_idx) in enumerate(skf.split(X_arr, y)):
    print(f'\n══════════ FOLD {fold+1}/{N_SPLITS} ══════════')

    X_tr, X_val = X_arr[trn_idx], X_arr[val_idx]
    y_tr, y_val = y[trn_idx],     y[val_idx]

    # XGBoost
    model_xgb = XGBClassifier(**xgb_params)
    model_xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb[val_idx] = model_xgb.predict_proba(X_val)
    test_xgb += model_xgb.predict_proba(test_arr) / N_SPLITS
    score = balanced_accuracy_score(y_val, np.argmax(oof_xgb[val_idx], axis=1))
    fold_scores_xgb.append(score)
    print(f'  XGB  BAcc={score:.5f}')

    # LightGBM
    import lightgbm as lgbm_lib
    model_lgb = LGBMClassifier(**lgb_params)
    model_lgb.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgbm_lib.early_stopping(stopping_rounds=100, verbose=False),
            lgbm_lib.log_evaluation(period=-1)
        ]
    )
    oof_lgb[val_idx] = model_lgb.predict_proba(X_val)
    test_lgb += model_lgb.predict_proba(test_arr) / N_SPLITS
    score = balanced_accuracy_score(y_val, np.argmax(oof_lgb[val_idx], axis=1))
    fold_scores_lgb.append(score)
    print(f'  LGB  BAcc={score:.5f}')

    # CatBoost
    from catboost import Pool
    train_pool = Pool(X_tr, y_tr)
    val_pool   = Pool(X_val, y_val)
    model_cat  = CatBoostClassifier(**cat_params)
    model_cat.fit(train_pool, eval_set=val_pool, early_stopping_rounds=100)
    oof_cat[val_idx] = model_cat.predict_proba(X_val)
    test_cat += model_cat.predict_proba(test_arr) / N_SPLITS
    score = balanced_accuracy_score(y_val, np.argmax(oof_cat[val_idx], axis=1))
    fold_scores_cat.append(score)
    print(f'  CAT  BAcc={score:.5f}')

print('\n══════════════════════════════════════════')
print(f'XGB  mean OOF: {np.mean(fold_scores_xgb):.5f} ± {np.std(fold_scores_xgb):.5f}')
print(f'LGB  mean OOF: {np.mean(fold_scores_lgb):.5f} ± {np.std(fold_scores_lgb):.5f}')
print(f'CAT  mean OOF: {np.mean(fold_scores_cat):.5f} ± {np.std(fold_scores_cat):.5f}')


══════════ FOLD 1/5 ══════════
  XGB  BAcc=0.96163
  LGB  BAcc=0.96723
  CAT  BAcc=0.95702

══════════ FOLD 2/5 ══════════
  XGB  BAcc=0.96094
  LGB  BAcc=0.96687
  CAT  BAcc=0.95658

══════════ FOLD 3/5 ══════════
  XGB  BAcc=0.96173
  LGB  BAcc=0.96764
  CAT  BAcc=0.95812

══════════ FOLD 4/5 ══════════
  XGB  BAcc=0.96248
  LGB  BAcc=0.96828
  CAT  BAcc=0.95845

══════════ FOLD 5/5 ══════════
  XGB  BAcc=0.96128
  LGB  BAcc=0.96630
  CAT  BAcc=0.95790

══════════════════════════════════════════
XGB  mean OOF: 0.96161 ± 0.00051
LGB  mean OOF: 0.96726 ± 0.00067
CAT  mean OOF: 0.95761 ± 0.00070


## 8. OOF-Weighted Ensemble

In [25]:
import numpy as np
from sklearn.metrics import balanced_accuracy_score

best_score = 0
best_w = None

# search weights
for w_lgb in np.arange(0.4, 0.8, 0.05):
    for w_xgb in np.arange(0.1, 0.5, 0.05):
        w_cat = 1 - w_lgb - w_xgb

        if w_cat < 0:
            continue

        oof_ensemble = (
            w_xgb * oof_xgb +
            w_lgb * oof_lgb +
            w_cat * oof_cat
        )

        preds = np.argmax(oof_ensemble, axis=1)
        score = balanced_accuracy_score(y, preds)

        if score > best_score:
            best_score = score
            best_w = (w_xgb, w_lgb, w_cat)

print("BEST WEIGHTS:", best_w)
print("BEST OOF SCORE:", best_score)

BEST WEIGHTS: (np.float64(0.20000000000000004), np.float64(0.75), np.float64(0.04999999999999996))
BEST OOF SCORE: 0.9663666533445383


In [26]:
w_xgb, w_lgb, w_cat = best_w

test_probs = (
    w_xgb * test_xgb +
    w_lgb * test_lgb +
    w_cat * test_cat
)

print(f"Final weights — XGB:{w_xgb:.3f}, LGB:{w_lgb:.3f}, CAT:{w_cat:.3f}")

Final weights — XGB:0.200, LGB:0.750, CAT:0.050


In [29]:
final_probs = (
    0.20 * test_xgb +
    0.75 * test_lgb +
    0.05 * test_cat
)

In [31]:
final_probs = (
    0.25 * test_xgb +
    0.75 * test_lgb
)

In [32]:
final_probs = test_lgb

In [3]:
# =========================
# FINAL ENSEMBLE (SUBMISSION)
# =========================

final_probs = (
    0.20 * test_xgb +
    0.75 * test_lgb +
    0.05 * test_cat
)

final_preds = np.argmax(final_probs, axis=1)

NameError: name 'test_xgb' is not defined

In [2]:
# Final ensemble
final_probs = (
    0.20 * test_xgb +
    0.75 * test_lgb +
    0.05 * test_cat
)

# Convert probabilities → predicted class
final_preds = np.argmax(final_probs, axis=1)

# Create submission
submission = pd.DataFrame({
    "id": test["id"],
    "class": final_preds
})

# Save file
submission.to_csv("submission.csv", index=False)

NameError: name 'test_xgb' is not defined

In [38]:
print(submission.head())

print("\nSubmission shape:")
print(submission.shape)

print("\nPrediction distribution:")
print(submission["class"].value_counts())

print("\nMissing values:")
print(submission.isnull().sum())

print("\nColumn types:")
print(submission.dtypes)

In [1]:
submission["class"].value_counts()

NameError: name 'submission' is not defined

In [ ]:
final_preds = np.argmax(final_probs, axis=1)

In [ ]:
import numpy as np

print("Class distribution:")
print(np.bincount(submission["class"]))

In [ ]:
print("Train distribution:")
print(np.bincount(y))

print("Test prediction distribution:")
print(np.bincount(final_preds))

In [ ]:
print(submission.isnull().sum())

In [ ]:
print(submission.dtypes)